## Image segmentation with SAM 3

This notebook demonstrates how to use SAM 3 for image segmentation with text or visual prompts. It covers the following capabilities:

- **Text prompts**: Using natural language descriptions to segment objects (e.g., "person", "face")
- **Box prompts**: Using bounding boxes as exemplar visual prompts

In [ ]:
import os
import gc
import sys
import cv2
sys.path.insert(0, "/home/groups/sammer/haogeh/util/models/sam3/")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
import sam3
from PIL import Image
from sam3 import build_sam3_image_model
from sam3.model.box_ops import box_xywh_to_cxcywh
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import draw_box_on_image, normalize_bbox, plot_results

sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")

In [ ]:
import importlib
importlib.reload(sam3)

In [ ]:
import torch

# turn on tfloat32 for Ampere GPUs
# https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

# Build Model

In [ ]:
bpe_path = f"{sam3_root}/assets/bpe_simple_vocab_16e6.txt.gz"
model = build_sam3_image_model(
    bpe_path=bpe_path,
    checkpoint_path = f'{sam3_root}/assets/checkpoint/sam3.pt',
    enable_inst_interactivity=True)

In [ ]:
dataset_path = os.path.join(sam3_root,'assets/images/dataset_basal_view_consented')
bbox_file = os.path.join(dataset_path,"df_with_nostril_verified.pkl")

In [ ]:
bbox_df = pd.read_pickle(bbox_file)

In [ ]:
def load_mask_dict(mask_path):
    mask_npz = np.load(mask_path,allow_pickle=True)
    mask_dict = {}
    mask_dict['bbox_xyxy'] = mask_npz['bbox_xyxy']
    mask_dict['bbox_xywh'] = mask_npz['bbox_xywh']
    mask_shape = mask_npz['shape']
    mask_dict['mask'] = np.unpackbits(mask_npz['mask'])[:np.prod(mask_shape)].reshape(mask_shape).astype(bool)
    mask_dict['mask_logit'] = mask_npz['mask_logit']
    mask_dict['score'] = mask_npz['score']
    return mask_dict

def load_mask_dict(mask_path):
    mask_npz = np.load(mask_path,allow_pickle=True)
    mask_dict = {}
    for key in mask_npz.keys():
        if key.endswith('mask'):
            mask_dict[key] = np.unpackbits(mask_npz[key])[:np.prod(mask_npz['shape'])].reshape(mask_npz['shape']).astype(bool)
        else:
            mask_dict[key] = mask_npz[key]
    return mask_dict

In [ ]:
'l_mask'.split('_')

In [ ]:
temp = np.load(nostril_mask_path,allow_pickle=True)

In [ ]:
def xyxy_to_xywh(bbox):
    """
    Convert a bounding box from (x1, y1, x2, y2) format to (x, y, w, h) format.
    (x, y) is the top-left corner, (w, h) is width and height.
    """
    x1, y1, x2, y2 = bbox
    x = x1
    y = y1
    w = x2 - x1
    h = y2 - y1
    return (x, y, w, h)

In [ ]:
from skimage.morphology import remove_small_objects

def extract_nose_image(image,mask,x1,y1,x2,y2):
    left, top, right, bottom = int(x1), int(y1), int(x2), int(y2)

    nose_image_raw = image.crop((left, top, right, bottom))
    mask_crop = mask[top:bottom,left:right]

    # Crop nose
    white_bg = Image.new("RGB", nose_image_raw.size, (255, 255, 255))
    mask_pil = Image.fromarray(mask_crop.astype('uint8') * 255)
    nose_image_masked = Image.composite(nose_image_raw, white_bg, mask_pil)
    return nose_image_raw, nose_image_masked
    
def get_nostril_mask(nose_image):
    lower = 0
    upper = np.percentile(nose_image,10)
    nose_image_np = np.array(nose_image)
    nose_image_gray = cv2.cvtColor(nose_image_np,cv2.COLOR_RGB2GRAY)
    mask = cv2.inRange(nose_image_gray,lower,upper)

    kernel = np.ones((5,5), np.uint8)
    mask = cv2.erode(mask, kernel, iterations=1)
    mask = cv2.dilate(mask, kernel, iterations=1)
    
    mask = (mask > 0)
    mask = remove_small_objects(mask, min_size=100)
    mask = (mask.astype(np.uint8) * 255)

    # Find connected components and their areas
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8))
    
    # Get areas for each component (excluding background which is label 0)
    areas = stats[1:, cv2.CC_STAT_AREA]
    
    
    # Get indices of top 2 largest components
    if len(areas) >= 2:
        top_2_indices = np.argsort(-areas)[:2] + 1  # Add 1 since we excluded background
        components = [labels == i for i in top_2_indices]
        medoids = []
        for component in components:
            ys, xs = np.nonzero(component)
            cx, cy = xs.mean(), ys.mean()          # centroid (float)
            pts = np.column_stack([xs, ys]).astype(np.float32)
            d2 = (pts[:,0]-cx)**2 + (pts[:,1]-cy)**2
            px, py = pts[np.argmin(d2)].astype(int)
            medoids.append((px, py))
        medoids = np.array(medoids)
        # Sort by x-coordinate to get left and right
        r_nostril_medoid, l_nostril_medoid = sorted(medoids, key=lambda x: x[0])

        mask = np.isin(labels, top_2_indices).astype(np.uint8) * 255
    else:
        print(f"ERROR: Only {len(areas)} components found in {image_path}")
        return mask, None, None

    return mask, l_nostril_medoid, r_nostril_medoid
def normalize_point(point,image):
    x,y = point
    width,height = image.size
    return x/width,y/height

In [ ]:
processor = Sam3Processor(model, confidence_threshold=0.5)

In [ ]:
def predict_inst(model,processor,inference_state,input_points,input_labels,input_bbox):
    processor.reset_all_prompts(inference_state)
    masks, scores, logits = model.predict_inst(
        inference_state,
        point_coords=input_points,      # numpy array, shape [2, 2]
        point_labels=input_labels,       # numpy array, shape [2]
        box=input_bbox,                  # numpy array, shape [4] (XYXY format)
        multimask_output=False,
    )

    mask = masks[0]
    score = scores[0]
    mask_logit = logits[0]
    return mask, score, mask_logit

In [ ]:
for i,line in bbox_df.iterrows():
    image_path = os.path.join(dataset_path,line['save_path_rel'])
    result_image_path = os.path.join(os.path.dirname(image_path), os.path.basename(image_path).split('.')[0] + '_result.jpg')

    nose_mask_path = os.path.join(os.path.dirname(image_path), os.path.basename(image_path).split('.')[0] + '_mask.npz')
    nostril_mask_path = os.path.join(os.path.dirname(image_path), os.path.basename(image_path).split('.')[0] + f'_nostril_mask.npz')
    # load: state = np.load("state.npz", allow_pickle=True)
    try:
        image = Image.open(image_path).convert("RGB")
    except:
        continue


    nose_bbox = line['bbox']
    nose_x1, nose_y1, nose_x2, nose_y2 = nose_bbox
    nose_x,nose_y,nose_w,nose_h = xyxy_to_xywh(nose_bbox)

    mask_dict = load_mask_dict(nose_mask_path)
    nose_mask = mask_dict['mask']

    nose_image_raw, nose_image_masked= extract_nose_image(image,nose_mask,nose_x1,nose_y1,nose_x2,nose_y2)
    nostril_mask, l_nostril_medoid, r_nostril_medoid = get_nostril_mask(nose_image_raw)

    nose_image = nose_image_raw
    width, height = nose_image.size

    inference_state = processor.set_image(nose_image)
    save_dict = {}

    l_nostril_bbox = line['left_nostril_bbox']
    r_nostril_bbox = line['right_nostril_bbox']

    l_input_points = np.array([l_nostril_medoid, r_nostril_medoid], dtype=np.float32)  # Shape: [2, 2]
    l_input_labels = np.array([1, 0], dtype=np.int32)  # Shape: [2]
    l_input_bbox = np.array(l_nostril_bbox, dtype=np.float32)  # Shape: [4]

    r_input_points = np.array([r_nostril_medoid, l_nostril_medoid], dtype=np.float32)  # Shape: [2, 2]
    r_input_labels = np.array([1, 0], dtype=np.int32)  # Shape: [2]
    r_input_bbox = np.array(r_nostril_bbox, dtype=np.float32)  # Shape: [4]

    l_mask, l_score, l_mask_logit = predict_inst(model,processor,inference_state,l_input_points,l_input_labels,l_input_bbox)
    r_mask, r_score, r_mask_logit = predict_inst(model,processor,inference_state,r_input_points,r_input_labels,r_input_bbox)
    
    fig, ax = plt.subplots()
    ax.imshow(l_mask)
    x1,y1,w,h = xyxy_to_xywh(l_nostril_bbox)
    rect = plt.Rectangle((x1, y1), w,h, edgecolor='red', facecolor='none', linewidth=1)
    ax.add_patch(rect)
    ax.scatter([l_nostril_medoid[0]],[l_nostril_medoid[1]],color = 'b', s = 10)
    ax.scatter([r_nostril_medoid[0]],[r_nostril_medoid[1]],color = 'g', s = 10)
    plt.show()
    plt.close()

    save_dict['shape'] = l_mask.shape
    save_dict['l_bbox'] = l_nostril_bbox
    save_dict['l_mask'] = np.packbits(l_mask.astype(bool))
    save_dict['l_score'] = l_score
    save_dict['l_mask_logit'] = l_mask_logit.astype(np.float16)

    save_dict['r_bbox'] = r_nostril_bbox
    save_dict['r_mask'] = np.packbits(r_mask.astype(bool))
    save_dict['r_score'] = r_score
    save_dict['r_mask_logit'] = r_mask_logit.astype(np.float16)

    np.savez_compressed(nostril_mask_path, **save_dict)

    # --- VISUALIZATION (Object-Oriented Approach) ---

    # Create explicit figure to avoid pyplot state leak
    mask_dict = load_mask_dict(nostril_mask_path)
    fig, ax = plt.subplots()
    ax.imshow(nose_image)
    
    for side in ['l','r']:
        x1,y1,x2,y2 = mask_dict[f'{side}_bbox']
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, edgecolor='r' if side=='l' else 'b', facecolor='none', linewidth=1)
        ax.add_patch(rect)
        mask = mask_dict[f'{side}_mask']
        mask_np = mask.astype(bool)
        overlay = np.zeros((mask_np.shape[0], mask_np.shape[1], 4), dtype=float)
        overlay[mask_np] = [0.0, 1.0, 0.0, 0.5]
        ax.imshow(overlay)

    ax.set_title(f"MRN: {line['mrn']}")
    ax.axis('off')
    fig.tight_layout()
    # Save using the figure object
    fig.savefig(result_image_path, bbox_inches='tight', pad_inches=0)
    
    # Explicitly close the specific figure object
    plt.close(fig) 
    

    # --- CLEANUP ---
    # Explicitly delete heavy tensors
    del inference_state
    
    # Periodically empty cache (e.g., every 10 images) rather than every image
    # to keep speed up, unless memory is extremely tight.
    if i % 5 == 0:
        gc.collect()
        torch.cuda.empty_cache()
